# Using the ZMART-controller

The controller offers a short, fixed list of commands, and every microscope answers them in the same form. A workflow written against these commands runs on any microscope that has a ZMART driver. This notebook goes through each command once, on the mock driver, a simulated microscope that is included with the controller and requires no hardware. Run the cells in order.

## Connecting

`get_instruments` lists the drivers installed on this computer, with the connection each one uses. The mock driver is always listed. `connect` opens the connection; from then on, every command is called as `mic.<command>`. The second argument is optional and is passed to the driver unchanged; here it tells the mock driver where to save its images.

In [1]:
from zmart_controller import mic

mic.get_instruments()

{'mock': {}}

In [2]:
mic.connect("mock", {"output_root": "images"})

## Registering a driver

The mock driver is included with the controller and is always listed. A real microscope needs its driver registered on the microscope computer, once, by pointing the controller at the driver's `zmart_driver.json`:

```python
mic.register_driver("C:/drivers/my-scope/zmart_driver.json")
```

The controller records the driver in its registry on that computer. From then on, in every Python session, `get_instruments` lists it and `connect` accepts its name. Part 2, [Plugging in a ZMART-driver](../2_plug_in_a_driver/README.md), describes the driver's files and how to write one.

## The microscope description

Every command answers with two fields: `success`, whether the driver carried out the request, and `content`, the information it returns. The content of `get_info` is a description of the microscope in plain language: what each setting means, its unit and bounds, and which objectives are fitted. It contains what the other commands cannot report, and is worth reading once for every new microscope.

In [3]:
print(mic.get_info()["content"]["description"])

A pretend widefield fluorescence microscope that runs entirely in software (MockScope Control), for trying workflows without hardware. It images a sample of scattered fluorescent spots on a slide that is slightly tilted, so the height at which the spots are sharp changes a little from place to place.

Stage: x and y move the slide, z moves the focus, all in micrometres from the origin. z has two motors: "motoric" for long moves and "piezo" for fine, fast steps around the current height.

Objectives, by slot: 1 is 10x/0.30 Air (1.0 µm per pixel), 2 is 20x/0.75 Air (0.5 µm per pixel), 3 is 40x/0.95 Air (0.25 µm per pixel). Every image is 64 x 64 pixels, so the field of view shrinks as the magnification grows. Changing objective shifts the view slightly; the driver corrects for it.

Settings (the changeable part of the state): objective is the slot number above; laser_power is the excitation in percent, 0 to 50; gain is the detector gain, 0 to 800; exposure_ms is the exposure time in mill

## Position

Positions are given in micrometres from the origin, a point chosen and saved once for this microscope, so that a position refers to the same place on the sample in every session. Each axis has one or more motors; on the mock driver, z has a coarse motor and a piezo for fine steps. The `canvas` of an axis is the range in which a picture can show anything: the travel of the stage plus half a field of view. Positions should be planned inside it.

In [4]:
mic.get_actuators()["content"]

{'x': ['motoric'], 'y': ['motoric'], 'z': ['motoric', 'piezo']}

In [5]:
mic.get_xyz()["content"]

{'x': {'value': 0.0, 'actuator': 'motoric', 'canvas': [-5032.0, 5032.0]},
 'y': {'value': 0.0, 'actuator': 'motoric', 'canvas': [-5032.0, 5032.0]},
 'z': {'value': 0.0, 'actuator': 'motoric', 'canvas': [-500.0, 500.0]}}

## Moving

`set_xyz` takes all three axes, so a script always states the full target position. When the answer is returned, the stage has arrived. A move outside the travel is not carried out: the answer is `success: False`, and `content` gives the reason. A workflow should check `success` before continuing.

In [6]:
mic.set_xyz(100, 50, 0)["content"]

{'position': {'x': 100, 'y': 50, 'z': 0},
 'readback': {'x': 100.0, 'y': 50.0, 'z': 0.0},
 'actuators': {'x': 'motoric', 'y': 'motoric', 'z': 'motoric'}}

In [7]:
mic.set_xyz(99999, 0, 0)

move refused by the limits: x would go to 149999.00 µm (stage coordinates), outside the travel range [45000.0, 55000.0] set in the limits


{'success': False,
 'content': 'ValueError: move refused: x would go to 149999.00 µm (stage coordinates), outside the travel range [45000.0, 55000.0] set in the limits'}

## Settings

The state is a snapshot of the instrument settings. Its `changeable` part is what `set_state` can apply: on the mock driver, laser power, gain, exposure and the objective. Some or all of them can be sent; the others remain unchanged. The driver reads each setting back to confirm that it was applied, and the answer lists what was applied. A state is a plain dictionary, so it can be saved with `json` and applied again in a later session.

In [8]:
mic.get_state()["content"]["changeable"]

{'laser_power': 10.0, 'gain': 100.0, 'exposure_ms': 10.0, 'objective': 1}

In [9]:
mic.set_state({"changeable": {"exposure_ms": 20.0}})["content"]

{'applied': {'exposure_ms': 20.0}, 'unconfirmed': {}}

## Acquisition

Acquisition settings are the choices about how an image is captured and saved, as opposed to the instrument settings: the file format, or the planes of a z-stack. For each one, `options` lists the permitted values and `active` the value used when none is given.

`acquire` captures an image at the current position with the current settings and saves it, in one step. The label names this position in the saved files. The answer lists every saved file under `files`, and under `planes` records for every image plane its file, channel, depth and time point, and the stage position at which it was taken. These paths should be used as returned; where the files are stored is decided by the driver.

In [10]:
mic.get_acquisition_settings()["content"]

{'folder': {'options': 'any text; empty saves straight into output_root',
  'active': ''},
 'backlash_correction': {'options': [True, False], 'active': True},
 'format': {'options': ['ome-tiff', 'ome-zarr'], 'active': 'ome-tiff'},
 'z_planes': {'options': 'whole number from 1 up to the limit', 'active': 1},
 'z_step_um': {'options': 'number > 0', 'active': 1.0}}

In [11]:
answer = mic.acquire(position_label="A1")
answer["content"]["files"]

['images/A1.ome.tif', 'images/A1.commands.json']

A z-stack is a single acquisition with several planes. The mock driver starts at the current height and moves upwards; every plane records the height at which it was taken.

In [12]:
answer = mic.acquire(position_label="A2", acquisition_settings={"z_planes": 3, "z_step_um": 2.0})
[(plane["z"], plane["z_um"]) for plane in answer["content"]["planes"]]

[(0, 0.0), (1, 2.0), (2, 4.0)]

## Procedures

Procedures are routines offered by the microscope, such as autofocus. Each driver offers its own; the description states what a routine does and which options it accepts. A procedure is run by name, with its options given alongside the name.

In [13]:
mic.get_procedures()["content"]

{'autofocus': {'description': 'Take a short z-stack around the current height, find the sharpest plane, and move there. Optional: range_um (default 20), step_um (default 2).'},
 'backlash_takeup': {'description': 'Approach the current position from the same side every time, so that positions repeat exactly.'},
 'zero_piezo': {'description': 'Move the piezo to the middle of its range, keeping the focus height.'}}

In [14]:
mic.run_procedure({"name": "autofocus", "range_um": 10, "step_um": 1})["content"]

{'ran': 'autofocus',
 'focus_um': 5000.0,
 'z_um': 0.0,
 'scores': [45569606.0,
  75911678.0,
  127124417.0,
  208184159.0,
  303591538.0,
  352754216.0,
  314239453.0,
  222264661.0,
  137532208.0,
  81253737.0,
  48687555.0]}

## Disconnecting

`disconnect` closes the connection. To use the microscope again, call `connect` again.

In [15]:
mic.disconnect()

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.